# 06 — Final analysis (post-session, report material)

Runs **after** the single §0.7 test session, on the per-window CSVs it
wrote. **No test access of its own**: it reads files, never a checkpoint
and never the dataset, so it adds no line to `test_invocations.jsonl` and
can be re-run freely.

Covers the two zero-cost consolidation directions (`CONSOLIDATION_REVIEW.md`):

- **D — model characterization:** per-class and per-AR-set error structure,
  calibration (reliability + ECE). macro-F1 averages 8 classes equally and
  hides whether the model is uniformly mediocre or strong on 6 and blind on
  2; this is also where the declared "val is blind to C/E/S" caveat gets
  tested empirically — do those classes actually underperform on test?
- **A — paired comparison:** trace-level (cluster) paired bootstrap on the
  difference between two streams.

**Row-count agnostic:** it analyses whatever streams have CSVs, so a 13- or
14-row session both work without edits.

## Two methodological constraints, decided before seeing any number

**1. The bootstrap runs on ACCURACY, not macro-F1.** `harness.macro_f1`
averages only over classes present in the ground truth. On the primary test
(11 traces, 6 of 8 classes with a single trace each) a trace-level resample
contains all 8 classes only ~2.8% of the time (measured, 20k resamples), so
a macro-F1 bootstrap would average a *different class set* in ~97% of
replicates — the estimand changes replicate to replicate and the interval
means nothing. Accuracy is well defined whatever classes appear. macro-F1 is
still reported, descriptively, without an interval.

**2. This measures ONE of the two variances.** The bootstrap resamples the
*test set* with the trained models held fixed: it answers "how much does the
gap move if we had drawn different test traces?". It says nothing about
*training/seed* variance — the 5.45-pt C2 vs C2_s43 swing (E1′) lives there.
Report them as two separate, labelled uncertainties; conflating them would
overstate what either shows.

## Environment setup

No staging, no GPU, no checkpoints: this notebook only reads CSVs.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

import numpy as np
import pandas as pd

SEED = 42
N_BOOT = 10000  # fixed a priori

# Report-material output dirs. Every table the report cites is written to TAB_DIR
# as a CSV and every figure to FIG_DIR as a vector PDF, so nothing in the report
# is transcribed by hand from stdout (the report build reads these files).
TAB_DIR = REPO_DIR / "reports" / "tables"
FIG_DIR = REPO_DIR / "reports" / "figures"
print("Repo:", REPO_DIR)

## Analysis functions

Notebook-local by the project's dividing line (`notebooks/diagnostics/README.md`):
new math, but run once for the report rather than re-run across sessions. If
the team decides these numbers deserve package status, the move is
mechanical — same path NCM/kNN took into `sharp_har/diagnostics.py`.

In [ ]:
from math import comb


def per_class_report(df, labels):
    """Per-class precision/recall/F1 + support from a harness windows CSV.
    Classes absent from y_true are reported with support 0 (the declared
    val-blind classes are exactly these). `labels` must be THIS stream's
    own class list (C0 is 5-class/P1, the P2 streams 8-class) — the class
    index in y_true/y_pred is positional in that list."""
    rows = []
    for i, lab in enumerate(labels):
        tp = int(((df.y_pred == i) & (df.y_true == i)).sum())
        fp = int(((df.y_pred == i) & (df.y_true != i)).sum())
        fn = int(((df.y_pred != i) & (df.y_true == i)).sum())
        prec = tp / (tp + fp) if tp + fp else float("nan")
        rec = tp / (tp + fn) if tp + fn else float("nan")
        f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else float("nan")
        rows.append({"class": lab, "support": int((df.y_true == i).sum()),
                     "precision": prec, "recall": rec, "f1": f1})
    return pd.DataFrame(rows)


def calibration(df, labels, n_bins=10):
    """Reliability table + ECE from the fused per-class probabilities the
    harness stores as p_<label>. Confidence = max fused probability.
    Correctness uses the stored y_pred — the harness fusion decision — so
    the table stays consistent with the reported accuracy under BOTH
    fusions (under sharp_c0 the majority-vote y_pred is not the argmax of
    the mean softmax, so re-deriving the prediction here would diverge).
    ECE = sum_b (n_b/N) * |acc_b - conf_b| (equal-width bins)."""
    P = df[[f"p_{l}" for l in labels]].to_numpy(dtype=float)
    conf = P.max(axis=1)
    correct = (df.y_pred.to_numpy() == df.y_true.to_numpy()).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows, ece = [], 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if not m.any():
            continue
        acc_b, conf_b, n_b = correct[m].mean(), conf[m].mean(), int(m.sum())
        ece += (n_b / len(df)) * abs(acc_b - conf_b)
        rows.append({"bin": f"({lo:.1f},{hi:.1f}]", "n": n_b,
                     "confidence": conf_b, "accuracy": acc_b, "gap": acc_b - conf_b})
    return pd.DataFrame(rows), ece


def paired_bootstrap(df_a, df_b, n_boot=N_BOOT, seed=SEED):
    """Trace-level (cluster) PAIRED bootstrap of the accuracy difference
    A - B. Resamples TRACES with replacement (never windows: windows within
    a trace are correlated, and window-level resampling would contradict the
    split-by-trace principle and understate variance), and applies the SAME
    resample to both streams so per-trace difficulty cancels.

    Returns the observed difference, the percentile CI, and the fraction of
    replicates favouring A. Accuracy, not macro-F1 — see the header."""
    a = df_a.set_index(["trace_id", "window_start"]).sort_index()
    b = df_b.set_index(["trace_id", "window_start"]).sort_index()
    assert a.index.equals(b.index), (
        "the two streams do not cover the same fused windows — different split "
        "or set; a paired comparison is only defined on identical units"
    )
    ok_a = (a.y_pred == a.y_true).to_numpy()
    ok_b = (b.y_pred == b.y_true).to_numpy()
    traces = a.index.get_level_values(0).to_numpy()
    uniq = np.unique(traces)
    idx_of = {t: np.flatnonzero(traces == t) for t in uniq}

    observed = ok_a.mean() - ok_b.mean()
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for k in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        sel = np.concatenate([idx_of[t] for t in pick])
        diffs[k] = ok_a[sel].mean() - ok_b[sel].mean()
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"observed": observed, "ci_low": lo, "ci_high": hi,
            "frac_favouring_a": float((diffs > 0).mean()), "n_traces": len(uniq)}


def mcnemar(df_a, df_b):
    """Exact two-sided McNemar (sign test) on the window-level discordance of
    two streams over the SAME fused windows (aligned on (trace_id, window_start),
    same assert as paired_bootstrap). Returns (only_a, only_b, both, neither, p).

    ANTI-CONSERVATIVE by construction: windows of one trace are correlated, so
    this p treats them as independent. Read the counts as descriptive mechanism
    and let the trace-level bootstrap decide significance (G10)."""
    a = df_a.set_index(["trace_id", "window_start"]).sort_index()
    b = df_b.set_index(["trace_id", "window_start"]).sort_index()
    assert a.index.equals(b.index), (
        "the two streams do not cover the same fused windows — different split or set"
    )
    oa = (a.y_pred == a.y_true).to_numpy()
    ob = (b.y_pred == b.y_true).to_numpy()
    only_a, only_b = int((oa & ~ob).sum()), int((~oa & ob).sum())
    both, neither = int((oa & ob).sum()), int((~oa & ~ob).sum())
    n = only_a + only_b
    p = 1.0
    if n:
        kk = min(only_a, only_b)
        p = min(1.0, 2 * sum(comb(n, i) for i in range(kk + 1)) / (2 ** n))
    return only_a, only_b, both, neither, p


def top_confusions(cm, n=6):
    """Dominant off-diagonal confusions from a harness confusion DataFrame
    (index = true class, columns = predicted). Returns [(true, pred, count)]
    sorted by descending count — the 'are the errors physically sensible?' cut."""
    out = [(t, q, int(cm.loc[t, q])) for t in cm.index for q in cm.columns
           if t != q and cm.loc[t, q] > 0]
    return sorted(out, key=lambda r: -r[2])[:n]


def seen_blind_f1(df, labels, seen):
    """Class-coverage decomposition (G12): mean per-class F1 over the val-SEEN
    classes vs the val-BLIND ones (the classes checkpoint selection never saw).
    Returns dict(F1 val-seen, F1 val-blind, blind - seen). 'blind - seen' > 0
    means the val-blind classes do BETTER on test — i.e. the declared
    'val-blind classes underperform' caveat does NOT hold on that stream."""
    pc = per_class_report(df, labels).set_index("class")
    blind = [c for c in labels if c not in seen]
    f1_seen = pc.loc[seen, "f1"].mean()
    f1_blind = pc.loc[blind, "f1"].mean()
    return {"F1 val-seen": f1_seen, "F1 val-blind": f1_blind,
            "blind - seen": f1_blind - f1_seen}

## Self-test on synthetic data

Run this **before** the session: it proves the three functions behave on
inputs whose answers are known, so on session day the code is already
trusted. Same discipline as the day-2/day-3 synthetic verification.

In [ ]:
def _synth(n_traces=11, per_trace=40, n_cls=8, acc=0.8, seed=0):
    """Fake windows CSV: one activity per trace (as in the real test set).
    Confidence is pinned at 1 - 0.05*(n_cls-1) = 0.65 regardless of `acc`,
    which is what makes the calibration check below meaningful."""
    rng = np.random.default_rng(seed)
    labels = [chr(65 + i) for i in range(n_cls)]
    rows = []
    for t in range(n_traces):
        y = t % n_cls
        for w in range(per_trace):
            right = rng.random() < acc
            pred = y if right else int(rng.choice([c for c in range(n_cls) if c != y]))
            p = np.full(n_cls, 0.05); p[pred] = 1.0 - 0.05 * (n_cls - 1)
            rows.append({"trace_id": f"T{t}", "window_start": w * 340, "ar_set": "AR-7",
                         "y_true": y, "y_pred": pred,
                         **{f"p_{labels[c]}": p[c] for c in range(n_cls)}})
    return pd.DataFrame(rows), labels

df_hi, labels_s = _synth(acc=0.90, seed=1)
df_lo, _ = _synth(acc=0.60, seed=1)

# 1. per-class: support must sum to N, and a perfect stream must give F1 = 1
pc = per_class_report(df_hi, labels_s)
assert pc["support"].sum() == len(df_hi)
df_perf = df_hi.copy(); df_perf["y_pred"] = df_perf["y_true"]
assert np.allclose(per_class_report(df_perf, labels_s)["f1"].dropna(), 1.0)

# 2. calibration: ECE must track the |accuracy - confidence| GAP, not the
#    accuracy. Every synthetic sample sits at confidence 0.65, so all mass
#    is in one bin and ECE must equal |realized accuracy - 0.65| EXACTLY —
#    checked against the realized accuracy (not the nominal target, which
#    the finite sample misses by the Bernoulli noise). Both streams are
#    designed underconfident; the 90%-accurate one is the MORE miscalibrated
#    (gap ~0.25) despite the higher accuracy, the 60%-accurate one the better
#    matched (gap ~0.05). A naive implementation that conflates ECE with the
#    error rate flips the ordering and fails this.
_, ece_hi = calibration(df_hi, labels_s)
_, ece_lo = calibration(df_lo, labels_s)
acc_hi = (df_hi.y_pred == df_hi.y_true).mean()
acc_lo = (df_lo.y_pred == df_lo.y_true).mean()
assert abs(ece_hi - abs(acc_hi - 0.65)) < 1e-9, (ece_hi, acc_hi)
assert abs(ece_lo - abs(acc_lo - 0.65)) < 1e-9, (ece_lo, acc_lo)
assert ece_hi > ece_lo, (ece_hi, ece_lo)
assert acc_hi > 0.65, acc_hi  # the more-accurate stream is the underconfident one

# 3. paired bootstrap: A clearly better than B -> CI entirely above 0;
#    a stream against ITSELF -> observed 0 and a degenerate CI at 0
r_ab = paired_bootstrap(df_hi, df_lo)
r_aa = paired_bootstrap(df_hi, df_hi)
assert r_ab["observed"] > 0 and r_ab["ci_low"] > 0, r_ab
assert abs(r_aa["observed"]) < 1e-12 and r_aa["ci_low"] == r_aa["ci_high"] == 0.0, r_aa

# 4. mcnemar (G10): a stream vs itself -> zero discordance, p = 1, counts sum to N.
#    A stream that is right EXACTLY where another is wrong -> fully one-sided
#    discordance (only_b = 0) and an extreme p; counts still sum to N.
oa0, ob0, both0, neither0, p_self = mcnemar(df_hi, df_hi)
assert oa0 == ob0 == 0 and p_self == 1.0, (oa0, ob0, p_self)
assert both0 + neither0 == len(df_hi)
df_worse = df_perf.copy()
half = (df_worse.window_start // 340) % 2 == 0
df_worse.loc[half, "y_pred"] = (df_worse.loc[half, "y_true"] + 1) % len(labels_s)
oa1, ob1, both1, neither1, p1 = mcnemar(df_perf, df_worse)
assert ob1 == 0 and oa1 == int(half.sum()), (oa1, ob1)   # A right where B wrong, never the reverse
assert p1 < 1e-6 and oa1 + ob1 + both1 + neither1 == len(df_perf), (p1,)

# 5. top_confusions (direction D): known matrix -> diagonal excluded, sorted desc
cm_test = pd.DataFrame([[10, 3, 1], [2, 20, 0], [4, 0, 8]],
                       index=["A", "B", "C"], columns=["A", "B", "C"])
assert top_confusions(cm_test, n=2) == [("C", "A", 4), ("A", "B", 3)]

# 6. seen_blind_f1 (G12): seen classes perfect (F1=1), blind classes always
#    mispredicted to ANOTHER blind class (no FP leaks onto a seen class) -> the
#    val-seen mean is exactly 1, the val-blind mean 0, so blind - seen = -1.
df_sb, labels_sb = _synth(acc=1.0, seed=2)
mask_blind = df_sb.y_true.isin([5, 6, 7])
df_sb.loc[mask_blind, "y_pred"] = 5 + ((df_sb.loc[mask_blind, "y_true"] - 5 + 1) % 3)
sb = seen_blind_f1(df_sb, labels_sb, labels_sb[:5])
assert abs(sb["F1 val-seen"] - 1.0) < 1e-9 and abs(sb["F1 val-blind"]) < 1e-9, sb
assert abs(sb["blind - seen"] + 1.0) < 1e-9, sb

print("self-test passed")
print(f"  A(0.90) vs B(0.60): diff {r_ab['observed']:+.3f} "
      f"CI [{r_ab['ci_low']:+.3f}, {r_ab['ci_high']:+.3f}] over {r_ab['n_traces']} traces")
print(f"  ECE: 90%-accurate {ece_hi:.3f} (underconfident) | "
      f"60%-accurate {ece_lo:.3f} (closer to conf 0.65)")
print(f"  mcnemar one-sided: only_a {oa1}, only_b {ob1}, p {p1:.2e}  |  "
      f"seen-blind delta {sb['blind - seen']:+.2f}")

## Load the session's CSVs

Point `FINAL_DIR` at `reports/final/` (after the session commit) or at the
Drive run folders. Every `*_test_*_windows.csv` found becomes a stream.

**Naming contract with notebook 05.** The stream key is the filename prefix
*before* `_test_`, so notebook 05's final-copy step must emit files named
`<stream>_test_<fusion>_windows.csv` with a clean key — `C1`, `C2`, `C1_lin`,
`C3`, `C3_ft`, `C1_AdaBN`, `C1_T3A`, … — **not** the raw checkpoint stem
(`C1_best_test_…` would key the stream as `C1_best` and miss `FOCUS`/`PAIRS`).
The per-stream summary below prints every key and its class count precisely so
a stray stem or a class-set mismatch is caught before any analysis runs.

**Labels are per-stream, never global.** C0 is the 5-class P1 set and the P2
streams are 8-class with a different index encoding, so each stream is analysed
with its own `p_<label>` columns — mixing them would mislabel classes silently.

In [ ]:
FINAL_DIR = REPO_DIR / "reports" / "final"
assert FINAL_DIR.exists(), (
    f"{FINAL_DIR} not found — run this AFTER the §0.7 session has been committed "
    "(or point FINAL_DIR at the Drive run folders)."
)

streams = {}         # stream key -> windows DataFrame
stream_labels = {}   # stream key -> its OWN class list (C0 5-class, P2 8-class)
for p in sorted(FINAL_DIR.glob("*_windows.csv")):
    if "_test_" not in p.name:
        continue  # val CSVs, if any, stay out of the report analysis
    name = p.name.split("_test_")[0]
    df = pd.read_csv(p)
    streams[name] = df
    stream_labels[name] = [c[2:] for c in df.columns if c.startswith("p_")]

assert streams, f"no *_test_*_windows.csv under {FINAL_DIR}"
print(f"{len(streams)} streams (key = filename prefix before '_test_'; "
      "notebook 05 must emit '<stream>_test_<fusion>_windows.csv')")
for k, v in streams.items():
    acc = (v.y_pred == v.y_true).mean()
    print(f"  {k:24s} {len(v):5d} fused windows, {v.trace_id.nunique():3d} traces, "
          f"{len(stream_labels[k])} classes, acc {acc:.4f}")

In [ ]:
# Headline table for the report: accuracy (recomputed here) + macro-F1 (read from
# the harness metrics CSV's fused ALL row, NOT recomputed, so it stays identical to
# what the harness reported) + ECE (from calibration) + absent classes. macro-F1 is
# descriptive only — no CI (see the header: a trace-level macro-F1 bootstrap is
# ill-defined here). Written to TAB_DIR/summary.csv: this IS the report's main table,
# so it is read from a file, never retyped from stdout.
def _split_of(key):
    if key == "C0":
        return "P1 (5-cls)"
    if key.startswith("C1_s6out"):
        return "P2-living (S6-out)"
    return "P2-lab (S7-out)"

rows = []
for k in sorted(streams):
    df = streams[k]
    m = pd.read_csv(next(FINAL_DIR.glob(f"{k}_test_*_metrics.csv")))
    hd = m[(m.aggregation.astype(str).str.startswith("fused")) & (m.ar_set == "ALL")].iloc[0]
    av = hd.get("absent_classes", "")
    _, ece = calibration(df, stream_labels[k])
    rows.append({
        "stream": k, "split": _split_of(k),
        "n_traces": df.trace_id.nunique(), "n_win": len(df),
        "accuracy": (df.y_pred == df.y_true).mean(),
        "macro_f1": float(hd["macro_f1"]), "ECE": ece,
        "absent": "" if pd.isna(av) else str(av),
    })
summary = pd.DataFrame(rows)
TAB_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(TAB_DIR / "summary.csv", index=False)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nmacro-F1 descriptive (no CI). C0 is 5-class/P1 -> not comparable to the 8-class rows;")
print("C0 is a PARTIAL, NON-FAITHFUL reproduction (see the C0 cell) -> not a paper benchmark.")
print(f"This is the report's main results table -> {TAB_DIR / 'summary.csv'}")

## D — error structure and calibration

In [ ]:
FOCUS = "C1"  # the deliverable model; change to inspect another stream
assert FOCUS in streams, f"{FOCUS!r} not among loaded streams {sorted(streams)}"
df = streams[FOCUS]
labels = stream_labels[FOCUS]  # this stream's own class set (see the naming note)

print(f"=== {FOCUS}: per-class ===")
print(per_class_report(df, labels).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("Declared caveat to check here: val was blind to C/E/S -- do they underperform?")

print()
print(f"=== {FOCUS}: per-AR-set ===")
for s, g in df.groupby("ar_set"):
    print(f"  {s}: {len(g):5d} windows, acc {(g.y_pred == g.y_true).mean():.4f}")
print("(degenerate on the primary test -- a single AR-set; informative on C0 and S6-out)")

rel, ece = calibration(df, labels)
print()
print(f"=== {FOCUS}: calibration -- ECE {ece:.4f} ===")
print(rel.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("gap > 0 = underconfident, gap < 0 = overconfident in that bin")

In [ ]:
# Report figures. Vector PDFs into FIG_DIR (reports/figures/) -- copy them into the
# LaTeX report's figures/ folder. Confusion + per-class are drawn for every reported
# stream INCLUDING the backbone ablation (C1_sharplike) and the SupCon fine-tune
# (C3_ft), whose error structure the report discusses; the reliability overlay is a
# separate, narrower set (see below). Re-runnable, no test contact.
#   confusion_<stream>_test.pdf   row-normalized confusion (recall on the diagonal)
#   perclass_<stream>_test.pdf    per-class F1 bar
#   reliability_test.pdf          calibration overlay across loss families (same fusion)
FIG_STREAMS = [k for k in ("C0", "C1", "C2", "C3", "C3_ft", "C1_sharplike", "C1_s6out")
               if k in streams]
# Reliability compares CONFIDENCE vs accuracy, so every curve must share the SAME
# fusion. C0 uses sharp_c0 (majority VOTE): its "confidence" (max mean-softmax) is not
# P(y_pred), so it is EXCLUDED from the overlay (its ECE still appears in the ece table,
# with the footnote). The overlay is the softmax_avg loss-family contrast.
RELIABILITY_STREAMS = [k for k in ("C1", "C2", "C3", "C1_s6out") if k in streams]
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    FIG_DIR.mkdir(parents=True, exist_ok=True)

    def save_stream_figures(key):
        lab = stream_labels[key]
        # (a) confusion matrix, row-normalized (recall per true class on the diagonal)
        cm = pd.read_csv(next(FINAL_DIR.glob(f"{key}_test_*_confusion.csv")), index_col=0)
        M = cm.to_numpy(dtype=float)
        Mn = M / M.sum(axis=1, keepdims=True).clip(min=1)
        fig, ax = plt.subplots(figsize=(4.2, 3.8))
        im = ax.imshow(Mn, vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(range(len(cm.columns))); ax.set_xticklabels(cm.columns)
        ax.set_yticks(range(len(cm.index))); ax.set_yticklabels(cm.index)
        ax.set_xlabel("predicted"); ax.set_ylabel("true")
        ax.set_title(f"{key} — test confusion (row-normalized)")
        for i in range(Mn.shape[0]):
            for j in range(Mn.shape[1]):
                if Mn[i, j] > 0.005:
                    ax.text(j, i, f"{Mn[i, j]:.2f}", ha="center", va="center",
                            fontsize=7, color="white" if Mn[i, j] > 0.5 else "black")
        fig.colorbar(im, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"confusion_{key.lower()}_test.pdf"); plt.close(fig)

        # (b) per-class F1 bar (NaN -> 0 for absent classes; none absent on test)
        pc = per_class_report(streams[key], lab)
        fig, ax = plt.subplots(figsize=(4.2, 2.6))
        ax.bar(pc["class"], pc["f1"].fillna(0.0))
        ax.set_ylim(0, 1); ax.set_ylabel("F1"); ax.set_title(f"{key} — per-class F1 (test)")
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"perclass_{key.lower()}_test.pdf"); plt.close(fig)

    for key in FIG_STREAMS:
        save_stream_figures(key)

    # (c) reliability overlay -- the calibration contrast the ECE table quantifies:
    # CE rows (C1) systematically UNDER-confident (curve above the diagonal) from
    # label smoothing; C2/GRL closer to the diagonal. Same fusion only (C0 excluded).
    fig, ax = plt.subplots(figsize=(4.4, 4.0))
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="perfect calibration")
    for key in RELIABILITY_STREAMS:
        rel, ece = calibration(streams[key], stream_labels[key])
        if len(rel):
            ax.plot(rel["confidence"], rel["accuracy"], marker="o", ms=4,
                    label=f"{key} (ECE {ece:.3f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("confidence (max fused prob)"); ax.set_ylabel("accuracy")
    ax.set_title("Reliability (test) — above diagonal = underconfident")
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "reliability_test.pdf"); plt.close(fig)

    print("saved to", FIG_DIR)
    print("  confusion/perclass for", FIG_STREAMS)
    print("  reliability overlay (same fusion) for", RELIABILITY_STREAMS)
    print("-> copy these into the LaTeX report's figures/ folder.")
except ImportError:
    print("matplotlib not available -> skip figures (numbers above are unaffected).")

## A — paired bootstrap between streams

Only pairs on the **same split and set** are comparable, so the comparisons are
split into two groups: the P2-lab (S7-out) variants against the deliverable `C1`
— including the **backbone ablation** (`C1_sharplike`) and the augmentation
(`C1_aug`) — and, separately, the P2-living (S6-out) **pre-registered augmentation
test** (`C1_s6out_aug` vs `C1_s6out`). C0 (P1) is never paired with the P2
streams; the assert inside `paired_bootstrap` refuses mismatched units rather than
silently aligning. The seed (training) variance is reported in its own cell right
after — it is a different uncertainty and must not be merged with these CIs.

In [ ]:
# Paired bootstrap groups. Each pair MUST share the same split+set (the assert in
# paired_bootstrap refuses mismatched units). Direction: paired_bootstrap(A, B)
# reports A - B and P(A>B), so a positive diff favours A.

# PRE-REGISTERED primary comparisons (v5.2): only these two carry an inferential
# claim. Everything else is secondary/exploratory — its interval is uncorrected
# (G11 multiplicity, see the two-variance cell). Marked with * in the print and
# flagged in the persisted table so the report never calls a secondary "significant".
PRIMARY = {("C1", "C2"), ("C1", "C3")}

# --- primary rotation (P2-lab, S7-out, 8-class): everything vs the deliverable C1 ---
PAIRS = [
    ("C1", "C2"),            # GRL: does the adversary help? (expect C1 better)  [PRIMARY]
    ("C1", "C3"),            # SupCon                                            [PRIMARY]
    ("C1", "C3_ft"),         # SupCon fine-tune
    ("C1", "C1_sharplike"),  # BACKBONE ablation: our deep net vs the paper's shallow one
    ("C1", "C1_aug"),        # amplitude augmentation, in-rotation (S7)
    ("C1", "C1_AdaBN"),      # test-time BN
    ("C1", "C1_T3A"),        # test-time prototypes
    ("C1", "C1_AdaBN_T3A"),  # both TTA
]
# --- second rotation (P2-living, S6-out): the PRE-REGISTERED augmentation test ---
# A = the augmented model, so a positive diff means "augmentation helps".
PAIRS_S6 = [("C1_s6out_aug", "C1_s6out")]


def _run(pairs, title, group=""):
    """Runs the paired bootstrap for each pair, prints it, and RETURNS the rows
    (so they can be persisted / plotted). `resolved` = CI does not cross 0."""
    print(f"\n{title}")
    print(f"{'comparison':26s} {'diff':>8s} {'95% CI':>20s} {'P(A>B)':>8s}  flags")
    print("-" * 74)
    out = []
    for a, b in pairs:
        if a not in streams or b not in streams:
            print(f"{a + ' vs ' + b:26s}   (missing — skipped)")
            continue
        r = paired_bootstrap(streams[a], streams[b])
        resolved = bool(r["ci_low"] > 0 or r["ci_high"] < 0)
        primary = (a, b) in PRIMARY
        flags = ("*primary " if primary else "") + ("resolved" if resolved else "crosses-0")
        print(f"{a + ' vs ' + b:26s} {r['observed']:+8.4f} "
              f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}] {r['frac_favouring_a']:8.3f}  {flags}")
        out.append({"group": group, "comparison": f"{a} vs {b}", "A": a, "B": b,
                    "diff": r["observed"], "ci_low": r["ci_low"], "ci_high": r["ci_high"],
                    "P(A>B)": r["frac_favouring_a"], "resolved": resolved,
                    "primary": primary, "n_traces": r["n_traces"]})
    return out


_lab = _run(PAIRS, "=== P2-lab (S7-out): variants vs C1 ===", "P2-lab")
_s6 = _run(PAIRS_S6, "=== P2-living (S6-out): augmentation (A = augmented) ===", "P2-living")
paired_results = pd.DataFrame(_lab + _s6)
print("-" * 74)
print("CI crossing 0 = the accuracy gap is not resolved by this test sample.")
print("Only the two *primary rows carry an inferential claim (G11 multiplicity).")
print("This is TEST-SAMPLING variance only; seed variance is separate — see below.")

In [ ]:
# Persist the paired table (report source, with the resolved/primary flags) and draw
# the FOREST plot -- the single most report-ready figure from these numbers: every
# paired accuracy difference with its 95% CI, red line at 0, BOTH rotations (the
# P2-living augmentation test is pre-registered too). Plus the honest UNDER-POWER read:
# with only 11 traces on the primary rotation, "CI crosses 0" means "not resolvable
# here", NOT "no effect". We quantify the resolution floor rather than assert an effect.
TAB_DIR.mkdir(parents=True, exist_ok=True)
paired_results.to_csv(TAB_DIR / "paired_bootstrap.csv", index=False)
print(f"paired table -> {TAB_DIR / 'paired_bootstrap.csv'}")

lab = paired_results[paired_results.group == "P2-lab"]
resolved_gaps = lab.loc[lab.resolved, "diff"].abs()
half_widths = (lab.ci_high - lab.ci_low) / 2
print(f"\nUnder-power read (P2-lab, {int(lab.n_traces.iloc[0])} traces):")
print(f"  narrowest RESOLVED |gap|      {resolved_gaps.min():.3f}  "
      "(smaller gaps are NOT distinguishable from 0 on this test set)")
print(f"  median 95%-CI half-width      {half_widths.median():.3f}")
print("  => report unresolved gaps as 'not powered', a DIRECTION not a null result.")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    FIG_DIR.mkdir(parents=True, exist_ok=True)
    # Both rotations on one forest; reversed so PAIRS order reads top-to-bottom. The two
    # test sets differ in size (P2-lab n=11, P2-living n=15), so each row carries its
    # group tag -- they must not be read as one sample. * = pre-registered primary.
    fp = paired_results.iloc[::-1].reset_index(drop=True)
    ypos = np.arange(len(fp))
    colors = ["tab:blue" if p else "0.35" for p in fp["primary"]]
    fig, ax = plt.subplots(figsize=(5.8, 4.0))
    ax.hlines(ypos, fp["ci_low"], fp["ci_high"], colors=colors, linewidth=1.6)
    ax.scatter(fp["ci_low"], ypos, marker="|", c=colors, s=45)
    ax.scatter(fp["ci_high"], ypos, marker="|", c=colors, s=45)
    ax.scatter(fp["diff"], ypos, c=colors, s=24, zorder=3)
    ax.axvline(0, color="red", lw=1, ls="--")
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{c}{' *' if p else ''}  [{g}]"
                        for c, p, g in zip(fp["comparison"], fp["primary"], fp["group"])],
                       fontsize=7)
    ax.set_xlabel("accuracy difference (A − B), 95% CI  ·  * = pre-registered primary")
    ax.set_title("Paired bootstrap (test-sampling variance; P2-lab n=11, P2-living n=15)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "forest_paired.pdf"); plt.close(fig)
    print(f"forest figure -> {FIG_DIR / 'forest_paired.pdf'}")
except ImportError:
    print("matplotlib not available -> skip forest figure (table above is unaffected).")

In [ ]:
# Seed (training) variance -- the OTHER uncertainty, kept separate from the
# bootstrap above. The E1' seed twins as a plain spread of the test point
# estimates (this is NOT a bootstrap: n=2 seeds, so it is a spread, not a CI).
print("Seed spread on TEST (E1' twins) -- TRAINING variance, not test-sampling:")
for a, b in [("C1", "C1_s43"), ("C2", "C2_s43")]:
    if a in streams and b in streams:
        aa = (streams[a].y_pred == streams[a].y_true).mean()
        bb = (streams[b].y_pred == streams[b].y_true).mean()
        print(f"  {a:4s} vs {b:8s}: acc {aa:.4f} vs {bb:.4f}  ->  |Δ| {abs(aa - bb):.4f}")
print("\nRead alongside the bootstrap CIs: C1 is seed-stable, C2/GRL is not. If a")
print("same-config seed |Δ| rivals a cross-config gap, that gap is not seed-robust")
print("and must be reported as a direction with a range, not a point estimate.")

## E -- review follow-ups (NOTEBOOK_06_REVIEW.md: G2, G6, G7, G9-G12)

Added after the deep review of the real session. Everything here is analysis on
the already-committed `reports/final/` CSVs: **zero test contact**, re-runnable.

In [ ]:
# G12 -- class-coverage decomposition. The pre-declared caveat was "validation was
# blind to some classes, so those may underperform on test". This COMPUTES the
# answer (via seen_blind_f1, self-tested above) instead of eyeballing it. "F1
# val-seen" is also the val-visible macro-F1, i.e. the metric that checkpoint
# selection could actually observe. Persisted as the report's class-coverage table.
VAL_SEEN = {"p2_lab": ["H", "J", "L", "R", "W"], "p2_living": ["C", "E", "J", "R", "S"]}


def rotation_of(key):
    if key == "C0":
        return "p1"
    return "p2_living" if key.startswith("C1_s6out") else "p2_lab"


rows = []
for k in sorted(streams):
    rot = rotation_of(k)
    if rot == "p1":
        continue  # C0 is its own 5-class P1 split: no val-blind decomposition
    rows.append({"stream": k, "rotation": rot,
                 **seen_blind_f1(streams[k], stream_labels[k], VAL_SEEN[rot])})
coverage = pd.DataFrame(rows)
coverage.to_csv(TAB_DIR / "class_coverage.csv", index=False)
print(coverage.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print()
print("'blind - seen' > 0 means the val-BLIND classes do BETTER, i.e. the declared")
print("caveat does NOT hold there. The answer differs between rotations -- that")
print("non-uniformity is the reportable finding, not a single reassuring number.")
print(f"-> {TAB_DIR / 'class_coverage.csv'}")
print()
for k in ("C1", "C1_s6out"):
    if k in streams:
        print("per-class F1, " + k + " (" + rotation_of(k) + "):")
        print(per_class_report(streams[k], stream_labels[k])[["class", "support", "f1"]]
              .to_string(index=False, float_format=lambda x: f"{x:.3f}"))
        print()

In [ ]:
# G7 -- calibration across ALL streams (the FOCUS cell computes ECE for one stream).
# Persisted as the report's calibration table; the reliability overlay figure is in
# the direction-D figure cell above.
rows = []
for k in sorted(streams):
    _, e = calibration(streams[k], stream_labels[k])
    rows.append({"stream": k,
                 "accuracy": (streams[k].y_pred == streams[k].y_true).mean(),
                 "ECE": e})
ece_tbl = pd.DataFrame(rows).sort_values("ECE").reset_index(drop=True)
ece_tbl.to_csv(TAB_DIR / "ece.csv", index=False)
print(ece_tbl.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"-> {TAB_DIR / 'ece.csv'}")
print()
# Computed evidence for the label-smoothing reading (not just an assertion): in the
# CE deliverable C1 EVERY populated reliability bin has accuracy > confidence.
rel_c1, _ = calibration(streams["C1"], stream_labels["C1"])
under = int((rel_c1["gap"] > 0).sum())
print(f"C1 (CE): {under}/{len(rel_c1)} populated bins have accuracy > confidence "
      f"(mean gap {rel_c1['gap'].mean():+.3f}) -> systematic UNDER-confidence,")
print("the expected footprint of CE label smoothing capping the max softmax (not a bug).")
print("CAVEAT (C0): under sharp_c0 fusion y_pred is a majority VOTE, not the argmax of")
print("the mean softmax, so 'confidence' is not P(predicted class) -- C0's ECE is only")
print("indicative and needs a footnote if it appears in the report.")

In [ ]:
# G9 + G11 + G2 -- the two uncertainties SIDE BY SIDE, plus the multiplicity caveat.
def _acc(k):
    return (streams[k].y_pred == streams[k].y_true).mean()


print("Seed pairs as mean +/- half-range (n=2 seeds -> a RANGE, never a CI):")
for base, twin in [("C1", "C1_s43"), ("C2", "C2_s43")]:
    if base in streams and twin in streams:
        a, b = _acc(base), _acc(twin)
        print(f"  {base:3s}: {(a + b) / 2:.4f} +/- {abs(a - b) / 2:.4f}")
print()
if all(k in streams for k in ("C1", "C2", "C1_s43", "C2_s43")):
    r = paired_bootstrap(streams["C1"], streams["C2"])
    g42, g43 = _acc("C1") - _acc("C2"), _acc("C1_s43") - _acc("C2_s43")
    print("The headline C1 - C2 gap under BOTH uncertainties:")
    print(f"  seed 42 gap {g42:+.4f} | seed 43 gap {g43:+.4f}")
    print(f"  seed range of the gap      [{min(g42, g43):+.4f}, {max(g42, g43):+.4f}]")
    print(f"  test-sampling 95% CI (s42) [{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]")
    print()
    print("  Direction stable across seeds, but the test-sampling interval crosses 0.")
    print("  -> report a DIRECTION WITH A RANGE, not a point estimate.")
print()
print("G11 multiplicity: ~10 paired comparisons share the same 11 traces. The")
print("pre-registered PRIMARY comparisons are C1 vs C2 and C1 vs C3; the others are")
print("secondary/exploratory and their intervals are uncorrected. With n=11 clusters a")
print("formal correction would erase what little power exists, so we DECLARE the issue")
print("rather than apply one -- and never call a secondary comparison 'significant'.")

In [ ]:
# G6 -- per-trace correctness. On a single-AR-set test (primary S7) the TRACE is the
# granularity per-AR-set cannot provide. STRUCTURAL CONFOUND to declare: 6 of 8
# classes are a single trace here, so per-trace accuracy is ~ per-class recall and the
# trace-level bootstrap is effectively resampling the ACTIVITY MIX -- read the per-trace
# grid and the per-class report as near-redundant, not independent evidence.
# G10 -- window-level discordance (mcnemar, self-tested above; anti-conservative).
# (GRID_STREAMS is a distinct name from cell-14's PRIMARY set of pre-registered pairs.)
GRID_STREAMS = [k for k in ("C1", "C2", "C3", "C1_sharplike") if k in streams]
if GRID_STREAMS:
    base = streams[GRID_STREAMS[0]]
    grid = pd.DataFrame(index=sorted(base.trace_id.unique()))
    grid.index.name = "trace_id"
    for k in GRID_STREAMS:
        d = streams[k]
        ok = (d.y_pred == d.y_true)
        grid[k] = d.assign(_ok=ok).groupby("trace_id")["_ok"].mean()
    grid["class"] = base.groupby("trace_id").y_true.first().map(
        lambda i: stream_labels[GRID_STREAMS[0]][i])
    grid.to_csv(TAB_DIR / "per_trace_accuracy.csv")
    print("Per-trace accuracy, primary rotation (each trace ~ one class -- see note):")
    print(grid.to_string(float_format=lambda x: f"{x:.3f}"))
    print(f"-> {TAB_DIR / 'per_trace_accuracy.csv'}")
    print()


print("Discordance (window level, McNemar exact two-sided):")
for a_key, b_key in [("C1", "C3"), ("C1", "C2"), ("C1", "C1_T3A")]:
    if a_key in streams and b_key in streams:
        oa_, ob_, both, neither, pv = mcnemar(streams[a_key], streams[b_key])
        print(f"  {a_key} vs {b_key}: only-{a_key} {oa_}, only-{b_key} {ob_}, "
              f"both {both}, neither {neither}, p={pv:.4g}")
print()
print("NOTE: McNemar treats windows as independent -- they are NOT (windows of one")
print("trace are correlated), so this p-value is ANTI-conservative. Read the counts as")
print("descriptive mechanism; let the trace-level bootstrap decide significance.")

In [ ]:
# Dominant off-diagonal confusions (do the errors look physically sensible?) via the
# self-tested top_confusions, plus the direction-D cuts for C0, whose per-AR-set is the
# only NON-degenerate one (the primary P2 test is a single AR-set).
def _load_cm(key):
    return pd.read_csv(next(FINAL_DIR.glob(f"{key}_test_*_confusion.csv")), index_col=0)


for k in ("C1", "C1_s6out"):
    if k in streams:
        print(k + " top confusions (true -> pred, windows):")
        print("   " + " | ".join(f"{t}->{q} {n_}" for t, q, n_ in top_confusions(_load_cm(k))))
        print()

if "C0" in streams:
    print("=== C0 direction-D cuts (P1 split: the only non-degenerate per-AR-set) ===")
    for s, g in streams["C0"].groupby("ar_set"):
        print(f"  {s}: {len(g):5d} windows, acc {(g.y_pred == g.y_true).mean():.4f}")
    print()
    print(per_class_report(streams["C0"], stream_labels["C0"]).to_string(
        index=False, float_format=lambda x: f"{x:.3f}"))
    print()
    print("REPRODUCTION CAVEAT (C0): this is a PARTIAL, NON-FAITHFUL reproduction of")
    print("the paper (backbone / preprocessing / fusion differ), NOT a like-for-like")
    print("re-implementation. Read C0 as a sanity anchor that OUR harness runs the P1")
    print("5-class set end to end -- NOT as a benchmark vs the paper's number. The")
    print("report must not claim 'we reproduced X%'; it states a partial reproduction.")

## F -- report headlines (foregrounded syntheses)

The directions above compute everything; this section pulls the findings the report
leads with into one place -- so they are not buried as scattered rows. Still zero test
contact: it reads the same already-loaded streams and writes small CSVs to TAB_DIR.

In [ ]:
# F1 -- augmentation effect is ENVIRONMENT-DEPENDENT (headline). The SAME amplitude
# augmentation is tested in-rotation on both splits; it must be read per rotation, not
# pooled. diff = augmented - baseline (paired bootstrap), so > 0 means augmentation helps.
aug_pairs = [("P2-lab", "C1_aug", "C1"), ("P2-living", "C1_s6out_aug", "C1_s6out")]
rows = []
for grp, a, b in aug_pairs:
    if a in streams and b in streams:
        r = paired_bootstrap(streams[a], streams[b])
        rows.append({"rotation": grp, "comparison": f"{a} vs {b}", "diff": r["observed"],
                     "ci_low": r["ci_low"], "ci_high": r["ci_high"],
                     "resolved": bool(r["ci_low"] > 0 or r["ci_high"] < 0),
                     "n_traces": r["n_traces"]})
aug = pd.DataFrame(rows)
aug.to_csv(TAB_DIR / "augmentation_effect.csv", index=False)
print(aug.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"-> {TAB_DIR / 'augmentation_effect.csv'}")
if len(aug) == 2 and aug["diff"].iloc[0] * aug["diff"].iloc[1] < 0:
    print(f"\nHEADLINE: the augmentation diff FLIPS SIGN across environments "
          f"({aug['diff'].iloc[0]:+.3f} on lab vs {aug['diff'].iloc[1]:+.3f} on living).")
    print("Its value is environment-dependent -- do NOT report a single 'augmentation")
    print("helps/hurts' verdict; report the interaction with the held-out environment.")

In [ ]:
# F2 -- GRL damage localization (headline). Frozen features + a fresh linear probe vs
# the network's own trained head, SAME split+set so the pair is valid. diff = probe -
# end-to-end: ~0 means the trained head is already optimal on the representation;
# clearly > 0 means the head is the bottleneck (representation fine, damage in the
# classifier). This is the mechanism behind the C1-vs-C2 gap.
probe_rows = _run([(a, b) for a, b in [("C1_lin", "C1"), ("C2_lin", "C2")]
                   if a in streams and b in streams],
                  "=== probe vs end-to-end (A = frozen-feature linear probe) ===")
_pd = {r["B"]: r["diff"] for r in probe_rows}
if "C1" in _pd and "C2" in _pd:
    print(f"\nHEADLINE: probe-minus-e2e is {_pd['C1']:+.3f} for C1 (CE) but {_pd['C2']:+.3f}")
    print("for C2 (GRL). The GRL REPRESENTATION is not the problem -- a fresh linear head")
    print("recovers most of the gap, so the adversary's harm concentrates in the trained")
    print("CLASSIFIER HEAD, not the features. Report this as the C2 mechanism.")

In [ ]:
# F3 -- transductive test-time adaptation verdict (headline). AdaBN (test BN stats),
# T3A (test-time prototypes), and both, each vs the C1 deliverable, paired bootstrap.
# diff = adapted - C1, so > 0 means adaptation helps.
tta_pairs = [("C1_AdaBN", "C1"), ("C1_T3A", "C1"), ("C1_AdaBN_T3A", "C1")]
rows = []
for a, b in tta_pairs:
    if a in streams and b in streams:
        r = paired_bootstrap(streams[a], streams[b])
        rows.append({"method": a, "diff": r["observed"], "ci_low": r["ci_low"],
                     "ci_high": r["ci_high"],
                     "resolved": bool(r["ci_low"] > 0 or r["ci_high"] < 0)})
tta = pd.DataFrame(rows)
tta.to_csv(TAB_DIR / "tta_effect.csv", index=False)
print(tta.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"-> {TAB_DIR / 'tta_effect.csv'}")
if len(tta):
    best = tta.loc[tta["diff"].idxmax()]
    print(f"\nHEADLINE: no transductive method resolves a gain (best is {best['method']} "
          f"at {best['diff']:+.3f});")
    print("AdaBN is net-negative. On this single-AR-set test, test-time adaptation does")
    print("not help -- report it as a null/negative result, not a win.")

In [ ]:
# F4 -- worst-trace drill-down (mechanism for the C1 - C2 gap). On a single-AR-set
# test the trace ~ the class, so the traces where C2 collapses vs C1 name the ACTIVITIES
# the adversary breaks. Per-trace accuracy, sorted by the C1 - C2 advantage.
if all(k in streams for k in ("C1", "C2")):
    lab_c1 = stream_labels["C1"]

    def _per_trace(k):
        d = streams[k]
        return d.assign(_ok=(d.y_pred == d.y_true)).groupby("trace_id")["_ok"].mean()

    cls = streams["C1"].groupby("trace_id").y_true.first().map(lambda i: lab_c1[i])
    drill = pd.DataFrame({"class": cls, "C1": _per_trace("C1"), "C2": _per_trace("C2")})
    drill["C1 - C2"] = drill["C1"] - drill["C2"]
    drill = drill.sort_values("C1 - C2", ascending=False)
    drill.to_csv(TAB_DIR / "worst_trace_c1_vs_c2.csv")
    print("Per-trace accuracy, C1 vs C2 (largest C1 advantage first):")
    print(drill.to_string(float_format=lambda x: f"{x:.3f}"))
    print(f"-> {TAB_DIR / 'worst_trace_c1_vs_c2.csv'}")
    top = drill.iloc[0]
    print(f"\nHEADLINE: the C1-C2 gap is not diffuse -- it concentrates on a few traces.")
    print(f"The single largest driver is trace {drill.index[0]} (class {top['class']}), "
          f"where C1 scores {top['C1']:.2f} vs C2 {top['C2']:.2f}. Since each trace is")
    print("~one activity here, the gap is really 'GRL breaks these specific activities'.")

## Archiving

Executed copy → `notebooks/runs/YYYY-MM-DD_final_analysis.ipynb` + the
numbers into the report. No test access happened here, so there is nothing
to log in `test_invocations.jsonl` — and re-running is free.